# Rates

Every flow takes a **rate**. A rate can be a plain number, but it can also be a parameter
that you set when you run the model, a value that changes over time, or a calculation that
uses either. This page shows each of these in turn:

1. `Param`, a named parameter;
2. `Time()`, the current time, and arithmetic on rates;
3. mathematical functions such as `exp`;
4. `defer`, for when you would rather write your own Python function;
5. `named`, which helps summer4 reuse work when a model is rebuilt;
6. tables of values, for a rate that is read from data.

The model is the SIR model from the previous pages, so that every change of rate can be seen
in the epidemic curve.

```{admonition} Attribution
:class: note

Parts of this page's prose are adapted from the
[summer2 documentation](https://summer2.readthedocs.io), BSD-2-Clause,
Copyright (c) 2022, monash-emu
([monash-emu/summer2](https://github.com/monash-emu/summer2)). The code is
written for summer4.
```

## Imports and a helper

The next cell loads what we need. It also defines `epidemic`, a small function that builds the
SIR model with whatever infection rate you give it, runs it for 100 days and returns the
infectious people over time. We will call it once for each rate on this page.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Time,
    TransitionFlow,
    defer,
    exp,
    named,
)
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
days = np.linspace(0.0, 100.0, 101)
plan = SavePlan(requests={"compartments": SaveRequest(Compartments())}, ts=days)


def epidemic(contact_rate, params=None):
    """Run the SIR model with this contact rate and return the infectious people."""
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            ForceOfInfection("infection", infectious=state["I"], contact_rate=contact_rate),
        )
    )
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.2))
    model.set_initial_population({state["S"]: 990.0, state["I"]: 10.0})
    run = model.compile().run(params or {}, t0=0.0, t1=100.0, dt=0.1, save=plan)
    return run["compartments"].select(state["I"]).to_pandas().iloc[:, 0]

## A number and a Param

The simplest rate is a number: `contact_rate=0.5`. A `Param("contact_rate")` is a placeholder
for a number that you give when you call `run`, so the same compiled model can be run with
different values. Here we pass the value in a dictionary. The two lines should lie on top of
each other.

In [ ]:
pd.DataFrame(
    {
        "contact_rate=0.5": epidemic(0.5),
        'Param("contact_rate") = 0.5': epidemic(Param("contact_rate"), {"contact_rate": 0.5}),
    }
).plot(
    title="A number and a Param give the same epidemic",
    labels={"index": "time (days)", "value": "infectious people"},
)

## Time and arithmetic

`Time()` stands for the current time in the model, in days. You can add, subtract, multiply,
divide and raise rates to powers, in any combination of numbers, `Param`s and `Time()`.

Here the contact rate starts at `Param("start")` and falls by 0.004 each day, so after 50 days
it is 0.2 lower. `maximum` stops it from falling below 0.1. The plot compares this epidemic
with one at the constant starting rate. The falling contact rate should give a lower peak.

In [ ]:
from summer4 import maximum

falling = maximum(Param("start") - 0.004 * Time(), 0.1)
pd.DataFrame(
    {
        "constant contact rate 0.5": epidemic(0.5),
        "falling contact rate": epidemic(falling, {"start": 0.5}),
    }
).plot(
    title="A contact rate that falls over time",
    labels={"index": "time (days)", "value": "infectious people"},
)

## Mathematical functions

summer4 provides `exp`, `log`, `sqrt`, `tanh`, `clip`, `maximum` and `minimum`, which work on
rates. NumPy functions such as `np.exp` and `np.sin` also work when they are applied to a
rate.

A contact rate that decays exponentially from 0.6 with a half-life of 20 days can be written
`0.6 * exp(-Time() * 0.035)`. A seasonal rate can use `np.sin`. The plot shows the infectious
people for both. The decaying rate should produce a lower peak than the constant rate. The seasonal rate, which rises and
falls every 30 days, should give a lower peak followed by a long tail.

In [ ]:
decaying = 0.6 * exp(-Time() * 0.035)
seasonal = 0.3 * (1.0 + 0.9 * np.sin(2.0 * np.pi * Time() / 30.0))
pd.DataFrame(
    {
        "exponential decay": epidemic(decaying),
        "seasonal": epidemic(seasonal),
    }
).plot(
    title="Contact rates built from mathematical functions",
    labels={"index": "time (days)", "value": "infectious people"},
)

## Your own Python function with `defer`

Sometimes the easiest way to say what you mean is an ordinary Python function. `defer` lets
you use one as a rate: write `defer(f)(arguments)`, where the arguments are the things your
function needs, such as `Time()` and `Param(...)`. summer4 calls your function with the value
of each argument, at every time step.

The function below gives a contact rate that is `high` before a lockdown starts, `low` while
the lockdown lasts, and `high` again afterwards. Use `jnp` (JAX's version of NumPy) for
calculations inside the function, as in the `jnp.where` below, which means "if, then,
otherwise" for numbers. The plot compares the lockdown epidemic with one at the high rate
throughout. The lockdown should cut the first wave short, and a smaller second wave should follow
when the lockdown ends.

In [ ]:
import jax.numpy as jnp


def lockdown_rate(t, high, low, start, length):
    """High contact rate, except between start and start + length days."""
    in_lockdown = (t >= start) & (t < start + length)
    return jnp.where(in_lockdown, low, high)


lockdown = defer(lockdown_rate)(
    Time(), Param("high"), Param("low"), Param("start"), Param("length")
)
lockdown_params = {"high": 0.5, "low": 0.15, "start": 10.0, "length": 30.0}
pd.DataFrame(
    {
        "no lockdown": epidemic(0.5),
        "lockdown from day 10 to day 40": epidemic(lockdown, lockdown_params),
    }
).plot(
    title="A contact rate written as a Python function",
    labels={"index": "time (days)", "value": "infectious people"},
)

## Look at the rate itself

To check a function, plot the numbers it gives. Because `lockdown_rate` is an ordinary
function, we can call it ourselves on every day. The line should be high, drop to 0.15 for
days 10 to 40, and return to 0.5.

In [ ]:
pd.DataFrame(
    {"contact rate": np.asarray(lockdown_rate(days, 0.5, 0.15, 10.0, 30.0))},
    index=pd.Index(days, name="time (days)"),
).plot(
    title="The lockdown contact rate",
    labels={"index": "time (days)", "value": "contact rate (per day)"},
)

## Giving a function a name

If you build the same model more than once, for example inside a loop, summer4 compiles it
again each time, unless it can tell that the function is the same. `named(f, "label")` gives
a function a label that summer4 uses to recognise it. Use a label only for a function that
always does the same thing. The cell runs the lockdown model again and gives the same curve.

In [ ]:
labelled = defer(named(lockdown_rate, "lockdown_rate"))(
    Time(), Param("high"), Param("low"), Param("start"), Param("length")
)
pd.DataFrame(
    {
        "defer(lockdown_rate)": epidemic(lockdown, lockdown_params),
        'defer(named(lockdown_rate, "lockdown_rate"))': epidemic(labelled, lockdown_params),
    }
).plot(
    title="A named function gives the same epidemic",
    labels={"index": "time (days)", "value": "infectious people"},
)

## A rate read from a table

When the rate comes from data, give summer4 the dates and the values. `Data(times, values)`
holds the table and `.interp()` turns it into a rate that joins the values with straight
lines. `.interp("step")` holds each value until the next date instead. Outside the table
the first and last values are held.

The table below says that contact fell from 0.5 on day 10 to 0.2 on day 40 and then recovered
to 0.4 on day 70. The first plot compares the two ways of joining the values and the second shows the
epidemic with each of them. The step version should give a higher peak, because
contact stays at 0.5 until day 40, while the linear version starts falling at day 10.

In [ ]:
from summer4.data import Data

survey = Data(times=[10.0, 40.0, 70.0], values=[0.5, 0.2, 0.4])
linear_rate = survey.interp()
step_rate = survey.interp("step")
pd.DataFrame(
    {
        "linear": np.interp(days, survey.times, survey.values),
        "step": pd.Series(survey.values, index=survey.times)
        .reindex(days, method="ffill")
        .bfill()
        .to_numpy(),
    },
    index=pd.Index(days, name="time (days)"),
).plot(
    title="Two ways to join the values in a table",
    labels={"index": "time (days)", "value": "contact rate (per day)"},
)

In [ ]:
pd.DataFrame(
    {"linear": epidemic(linear_rate), "step": epidemic(step_rate)}
).plot(
    title="Epidemics from a tabulated contact rate",
    labels={"index": "time (days)", "value": "infectious people"},
)

## One column per group: `TableInterp`

A table can have one column for each group of people, for example one death rate per age
group. `Data.table(times, values, over=property)` builds it, and `.interp()` returns a
`TableInterp` rate that gives each group its own column. The flow then applies each group's
rate to that group.

Here a small population has two age groups, and the older group has a death rate that rises
over 20 years. The plot should show the old group shrinking faster as its rate climbs.

In [ ]:
from summer4 import ExitFlow

age = Property("age", ("young", "old"))
age_map = PropertyMap.from_property(age)
deaths = Data.table(
    [0.0, 10.0, 20.0],
    pd.DataFrame({"young": [0.001, 0.001, 0.001], "old": [0.01, 0.03, 0.06]}),
    over=age,
)
aging = FlowModel(age_map)
aging.add_flow(ExitFlow("death", age.present(), deaths.interp()))
aging.set_initial_population({age["young"]: 1000.0, age["old"]: 1000.0})
aging_plan = SavePlan(
    requests={"compartments": SaveRequest(Compartments())}, ts=np.linspace(0.0, 20.0, 41)
)
aging_run = aging.compile().run({}, t0=0.0, t1=20.0, dt=0.1, save=aging_plan)
aging_run["compartments"].to_pandas().plot(
    title="People alive, with a death rate that varies by age group and year",
    labels={"index": "time (years)", "value": "people"},
)

## Things to try

- In the lockdown example, change `start`, `length` and `low` in the dictionary. How late can
  the lockdown start and still lower the peak?
- Write your own `defer` function for a contact rate that ramps up linearly for 30 days and
  then stays flat. Compare it with `survey.interp()`.
- Replace `survey.interp()` with `survey.interp("sigmoidal")`. How does the curve differ?
- Make the starting value in `falling` a `Param` and plot the epidemic for three values.